# 01 — Municipal abstention: exploratory spatial diagnostics

**Research notebook — no observed results or operational forecasts are supplied.** This notebook checks the normalized TSE/IBGE dataset, summarizes observed municipal turnout, and computes an exploratory global Moran's I when suitable geography is available.

Expected input tables and validation requirements are defined in [the data dictionary](../docs/data-dictionary.md). The municipal boundary vintage and TSE-to-IBGE crosswalk must be audited before interpreting spatial results. Run from the repository's `notebooks/` directory or repository root after `pip install -e ".[dev]"`.

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

from abstention_spatial.core import (
    validate_municipal_panel,
    historical_round_change,
)
from abstention_spatial.spatial import municipal_moran

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

panel_path = repo_root / "data/processed/municipal_panel.csv"
geometry_path = repo_root / "data/geospatial/municipalities_2025.shp"

for label, path in [("Municipal panel", panel_path), ("Municipal boundaries", geometry_path)]:
    print(f"{label}: {'available' if path.exists() else 'not yet provided'} — {path}")


## 1. Validate and summarize the municipality-level panel

Expected columns: `municipality_code` (seven-character IBGE code), `year`, `round`, `eligible`, `turnout`, `absent`. Covariates are documented separately. The project validator rejects duplicate keys and inconsistent counts. A missing file will be reported without inventing observations.

In [ ]:
panel = None
if not panel_path.exists():
    print("Provide a normalized CSV according to docs/data-dictionary.md to proceed.")
else:
    source = pd.read_csv(panel_path, dtype={"municipality_code": "string"})
    panel = validate_municipal_panel(source)
    print(f"Validated rows: {len(panel):,}")
    print(panel.groupby(["year", "round"]).size().rename("municipalities"))


In [ ]:
observed = None
if panel is not None:
    observed = panel.loc[(panel["year"] == 2022) & (panel["round"] == 2)].copy()
    if observed.empty:
        print("2022 second-round municipal data not available.")
    else:
        totals = observed[["eligible", "absent"]].sum()
        weighted_abstention = totals["absent"] / totals["eligible"]
        print(f"Observed 2022 second-round rate across covered municipalities: {weighted_abstention:.2%}")
        print(f"Municipalities covered: {len(observed):,}")


## 2. Join reviewed boundaries and examine global spatial autocorrelation

Join on municipality codes, preserving all covered turnout units. If geometries are missing or the Queen adjacency graph has islands, stop rather than silently dropping localities. Moran's I is descriptive and does not establish spatial causation.

In [ ]:
if observed is None or observed.empty or not geometry_path.exists():
    print("Spatial diagnostic skipped: observed data and/or boundaries are unavailable.")
else:
    geometry = gpd.read_file(geometry_path)
    if "municipality_code" not in geometry.columns and "CD_MUN" in geometry.columns:
        geometry = geometry.rename(columns={"CD_MUN": "municipality_code"})
    if "municipality_code" not in geometry.columns:
        raise ValueError("Boundary file must have municipality_code or CD_MUN.")
    geometry["municipality_code"] = geometry["municipality_code"].astype("string").str.zfill(7)
    if geometry["municipality_code"].duplicated().any():
        raise ValueError("Boundary file has duplicate municipality codes.")
    joined = observed.merge(
        geometry[["municipality_code", "geometry"]],
        on="municipality_code",
        how="left",
        validate="one_to_one",
    )
    missing = int(joined["geometry"].isna().sum())
    print(f"Unmatched geometries: {missing}")
    if missing:
        print("Resolve geographic coverage before computing Moran's I.")
    else:
        gdf = gpd.GeoDataFrame(joined, geometry="geometry", crs=geometry.crs)
        try:
            statistic = municipal_moran(gdf, "abstention_rate")
            print(f"Moran's I: {statistic.I:.4f}; permutation p-value: {statistic.p_sim:.4f}")
        except ValueError as exc:
            print(f"Spatial weights need review: {exc}")


## 3. Illustrative, *unvalidated* second-round baseline

If the normalized panel contains **2022 rounds 1 and 2** and **2026 round 1**, compute a historical round-change benchmark. The 2026 second-round labels are explicitly excluded, even if present in the input. The fixed shrinkage parameter is illustrative; it must be tuned and evaluated using older elections before forecast publication. The eligible-voter denominator is provisional.

In [ ]:
if panel is None:
    print("Baseline skipped: municipal panel missing.")
else:
    required_rounds = {(2022, 1), (2022, 2), (2026, 1)}
    available = set(zip(panel["year"], panel["round"]))
    if not required_rounds.issubset(available):
        print("Baseline skipped: required historical and current first-round rows are unavailable.")
    else:
        cutoff_panel = panel.loc[
            (panel["year"] < 2026) |
            ((panel["year"] == 2026) & (panel["round"] == 1))
        ].copy()
        forecast = historical_round_change(
            cutoff_panel, historical_year=2022, target_year=2026,
            shrinkage=20000,
        )
        national_rate = (
            forecast["predicted_absent"].sum()
            / forecast["eligible_assumption"].sum()
        )
        print(f"Illustrative baseline across covered municipalities: {national_rate:.2%}")
        print("NOT a validated forecast. No 80%/95% predictive intervals are estimated.")


## Next steps

- Build provenance-checked and reconciled TSE/IBGE ingestion.
- Resolve disconnected weight-graph components and inspect alternative `W`.
- Backtest historical change, SAR/SEM/SDM and hierarchical count models before publishing forecasts.
- Add 80% and 95% predictive intervals and freeze inputs/model version by forecast cutoff.

See [validation protocol](../docs/validation.md) and [2026 forecast protocol](../docs/forecasting-2026.md).